# ICS40125 - Laboratorio N°06

Cuarteto de Anscombe: por qué la visualización importa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

data = sns.load_dataset("anscombe")
data.head()

## 1. Gráfico de dispersión de cada grupo

In [ ]:
grupos = ['I', 'II', 'III', 'IV']
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for ax, g in zip(axs.ravel(), grupos):
    sub = data[data['dataset'] == g]
    ax.scatter(sub['x'], sub['y'])
    ax.set_title(f'Grupo {g}')
plt.tight_layout()
plt.show()
# I: relacion lineal. II: curva no lineal. III: lineal con un outlier.
# IV: casi todo vertical con un punto que domina.

## 2. Resumen estadístico por grupo

In [ ]:
# medias, desviaciones y correlacion son casi identicas en los 4 grupos
resumen = data.groupby('dataset').agg(
    x_media=('x', 'mean'),
    y_media=('y', 'mean'),
    x_std=('x', 'std'),
    y_std=('y', 'std')
)
resumen['correlacion'] = data.groupby('dataset').apply(
    lambda d: d['x'].corr(d['y'])
)
resumen

## 3. Regresión lineal por grupo (sklearn)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for ax, g in zip(axs.ravel(), grupos):
    sub = data[data['dataset'] == g]
    X = sub[['x']].values
    y = sub['y'].values

    modelo = LinearRegression().fit(X, y)
    y_pred = modelo.predict(X)

    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    ax.scatter(sub['x'], sub['y'])
    ax.plot(sub['x'], y_pred, color='red')
    ax.set_title(f'Grupo {g} | MSE={mse:.2f} | R2={r2:.2f} | b1={modelo.coef_[0]:.2f}')
plt.tight_layout()
plt.show()
# Las cuatro rectas son casi iguales aunque los datos son muy distintos.

## 4. Estrategia para mejorar el ajuste

In [ ]:
# Grupo II tiene forma de parabola: un modelo lineal no sirve.
# Estrategia: ajustar un polinomio de grado 2.
from numpy.polynomial import polynomial as P

sub = data[data['dataset'] == 'II']
x = sub['x'].values
y = sub['y'].values

coef = np.polyfit(x, y, 2)          # ajuste cuadratico
y_pred = np.polyval(coef, x)
print("R2 cuadratico grupo II:", round(r2_score(y, y_pred), 4))

xs = np.linspace(x.min(), x.max(), 100)
plt.scatter(x, y)
plt.plot(xs, np.polyval(coef, xs), color='green')
plt.title('Grupo II con ajuste cuadratico')
plt.show()

In [ ]:
# Grupo III: el problema es un outlier. Estrategia: quitarlo y reajustar.
sub = data[data['dataset'] == 'III'].copy()
# el outlier es el punto con y mas alto
sub_sin = sub[sub['y'] < sub['y'].max()]

X = sub_sin[['x']].values
y = sub_sin['y'].values
modelo = LinearRegression().fit(X, y)
print("R2 grupo III sin outlier:", round(r2_score(y, modelo.predict(X)), 4))
# Al remover el outlier el ajuste lineal mejora muchisimo.